# **Library package**

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
library(purrr)

# **Import data**

In [3]:
val_dat<-readRDS("val_res.rds")

# **Data Dictionary**

### Data Dictionary: Medication and Observation Tracking

| Variable | Data Type | Description |
| :--- | :--- | :--- |
| **file_id** | Integer | Unique identifier for each record. |
| **t_rifam** | Logical | Observed correct number of pills (from AI) for Rifampicin. |
| **g_rifam** | Logical | Ground truth number of pills from prescription for Rifampicin. |
| **t_pyra** | Logical | Observed correct number of pills (from AI) for Pyrazinamide. |
| **g_pyra** | Logical | Ground truth number of pills for Pyrazinamide. |
| **t_iso** | Logical | Observed correct number of pills (from AI) for Isoniazid. |
| **g_iso** | Logical | Ground truth number of pills for Isoniazid. |
| **t_etham** | Logical | Observed correct number of pills (from AI) for Ethambutol. |
| **g_etham** | Logical | Ground truth number of pills for Ethambutol. |
| **t_med** | Logical | Correctly observed all pills (from AI). |
| **g_med** | Logical | Ground truth number of all pills from presciption. |
| **t_place** | Logical | Observed placement (from AI) attribute. |
| **g_place** | Logical | Ground truth placement from human review. |
| **t_swollow** | Logical | Observed swallowing action (from AI). |
| **g_swollow** | Logical | Ground truth swallowing action from human review. |
| **t_protude** | Logical | Observed protrusion (from AI). |
| **g_protude** | Logical | Ground truth protrusion from human review. |
| **t_seq** | Logical | Observed sequence order (from AI). |
| **g_seq** | Logical | Ground truth sequence order from human review. |
| **t_move** | Logical | Correct observed for all movements (from AI). |
| **g_move** | Logical | Ground truth of all movements from human review. |

# **Run the codes**

In [4]:
vars <- c("rifam", "pyra", "iso", "etham", "med", "place",
          "swollow", "protude", "seq", "move")

# 2. Manual calculation function in percentages using Wilson Method
calc_diag_percent <- function(var_name, data) {

  test_val <- data[[paste0("t_", var_name)]]
  gold_val <- data[[paste0("g_", var_name)]]

  # Calculate 2x2 components
  tp <- sum(test_val == TRUE & gold_val == TRUE, na.rm = TRUE)
  fp <- sum(test_val == TRUE & gold_val == FALSE, na.rm = TRUE)
  fn <- sum(test_val == FALSE & gold_val == TRUE, na.rm = TRUE)
  tn <- sum(test_val == FALSE & gold_val == FALSE, na.rm = TRUE)

  # Helper for proportions to Percent with 95% CI (Wilson Method)
  get_ci_percent <- function(x, n) {
    if (n == 0) return("N/A")

    # prop.test performs the Wilson score interval
    # correct = FALSE provides the standard Wilson method
    res <- prop.test(x, n, conf.level = 0.95, correct = FALSE)

    # Extract values
    est <- round(res$estimate * 100, 1)
    low <- round(res$conf.int[1] * 100, 1)
    high <- round(res$conf.int[2] * 100, 1)

    return(sprintf("%.1f%% (%.1f%%, %.1f%%)", est, low, high))
  }

  # Create a row for this variable
  data.frame(
    Variable = var_name,
    Sensitivity = get_ci_percent(tp, tp + fn),
    Specificity = get_ci_percent(tn, tn + fp),
    PPV = get_ci_percent(tp, tp + fp),
    NPV = get_ci_percent(tn, tn + fn),
    Accuracy = get_ci_percent(tp + tn, tp + tn + fp + fn),
    stringsAsFactors = FALSE
  )
}

# 3. Apply to all variables
final_table_percent <- map_df(vars, ~calc_diag_percent(.x, val_dat))

Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”


# **Show the results**

In [5]:
print(final_table_percent)

   Variable            Sensitivity            Specificity
1     rifam   79.7% (72.4%, 85.5%)   50.0% (29.0%, 71.0%)
2      pyra   87.6% (80.0%, 92.6%)   94.6% (85.4%, 98.2%)
3       iso   88.2% (82.2%, 92.4%)   87.5% (52.9%, 97.8%)
4     etham   82.1% (73.7%, 88.2%)   58.2% (45.0%, 70.3%)
5       med   72.6% (62.9%, 80.6%)   86.4% (76.1%, 92.7%)
6     place   68.6% (61.0%, 75.3%) 100.0% (51.0%, 100.0%)
7   swollow   78.0% (70.9%, 83.7%) 100.0% (43.9%, 100.0%)
8   protude 100.0% (75.8%, 100.0%) 100.0% (97.5%, 100.0%)
9       seq   90.6% (84.7%, 94.5%)   90.9% (72.2%, 97.5%)
10     move 100.0% (56.6%, 100.0%)   99.4% (96.5%, 99.9%)
                      PPV                    NPV               Accuracy
1    92.7% (86.7%, 96.1%)   23.7% (13.0%, 39.2%)   76.4% (69.3%, 82.3%)
2    96.8% (91.1%, 98.9%)   80.3% (69.2%, 88.1%)   90.1% (84.5%, 93.8%)
3    99.3% (96.0%, 99.9%)   28.0% (14.3%, 47.6%)   88.2% (82.3%, 92.3%)
4    79.1% (70.6%, 85.6%)   62.7% (49.0%, 74.7%)   73.9% (66.6%, 80.1%)
5 

Then, calculate the overall verification performance for both TAL verification and pill count.

In [7]:
# 1. Helper function to extract and clean numeric values from your table
get_metrics <- function(df, row_name, col_name) {
  text <- df[df$Variable == row_name, col_name]
  # Extract all numbers (including decimals) using regex
  nums <- as.numeric(unlist(regmatches(text, gregexpr("[0-9.]+", text)))) / 100
  return(nums) # Returns vector: [Estimate, Lower, Upper]
}

# 2. Extract specific test data from final_table_percent
med_se  <- get_metrics(final_table_percent, "med", "Sensitivity")
med_sp  <- get_metrics(final_table_percent, "med", "Specificity")
move_se <- get_metrics(final_table_percent, "move", "Sensitivity")
move_sp <- get_metrics(final_table_percent, "move", "Specificity")

# Calculate Prevalence based on 'med' metrics to keep accuracy consistent
med_acc <- get_metrics(final_table_percent, "med", "Accuracy")[1]
prev    <- (med_acc - med_sp[1]) / (med_se[1] - med_sp[1])

# 3. Calculate Point Estimates (Parallel/OR Rule)
comb_se  <- med_se[1] + move_se[1] - (med_se[1] * move_se[1])
comb_sp  <- med_sp[1] * move_sp[1]
comb_acc <- (comb_se * prev) + (comb_sp * (1 - prev))

# 4. Deterministic Error Propagation (Delta Method)
# SE = (Upper - Lower) / 3.92
se_se1 <- (med_se[3] - med_se[2]) / 3.92
se_sp1 <- (med_sp[3] - med_sp[2]) / 3.92
se_se2 <- (move_se[3] - move_se[2]) / 3.92
se_sp2 <- (move_sp[3] - move_sp[2]) / 3.92

# Variance for Specificity (Product Rule: Var(AB))
var_sp <- (med_sp[1]^2 * se_sp2^2) + (move_sp[1]^2 * se_sp1^2)
se_comb_sp <- sqrt(var_sp)

# Variance for Sensitivity (OR Rule: Var(1 - (1-A)(1-B)))
var_se <- ((1 - med_se[1])^2 * se_se2^2) + ((1 - move_se[1])^2 * se_se1^2)
se_comb_se <- sqrt(var_se)

# Variance for Accuracy (Weighted Sum)
var_acc <- (prev^2 * var_se) + ((1 - prev)^2 * var_sp)
se_comb_acc <- sqrt(var_acc)

# 5. Result Formatting
parallel_results <- data.frame(
  Metric   = c("Sensitivity", "Specificity", "Accuracy"),
  Estimate = c(comb_se, comb_sp, comb_acc),
  Lower    = c(comb_se - 1.96 * se_comb_se, comb_sp - 1.96 * se_comb_sp, comb_acc - 1.96 * se_comb_acc),
  Upper    = c(comb_se + 1.96 * se_comb_se, comb_sp + 1.96 * se_comb_sp, comb_acc + 1.96 * se_comb_acc)
)

# Convert to percentage
parallel_results[,2:4] <- lapply(parallel_results[,2:4], function(x) x * 100)

# --- CONSTRAINT RULE ---
# Manually adjust Sensitivity Lower Bound to match the best individual test (move)
# This prevents the Delta Method from incorrectly showing 100% (100%-100%)
parallel_results[parallel_results$Metric == "Sensitivity", "Lower"] <- move_se[2] * 100

# Final cleanup: Cap all values at [0, 100]
parallel_results[,2:4] <- lapply(parallel_results[,2:4], function(x) pmin(pmax(x, 0), 100))

print(parallel_results)

       Metric  Estimate    Lower     Upper
1 Sensitivity 100.00000 56.60000 100.00000
2 Specificity  85.88160 77.50167  94.26153
3    Accuracy  94.16849 89.25320  99.08377
